# Chapter 11 — Constrained Optimization

Companion notebook to Chapter 11 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Convention of the chapter: at a constrained extremum
`df(a) + lambda o Dg(a) = 0`; on arrays, `gamma + J_g^T lambda = 0`.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, hessian

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

def newton_lagrange(f, g, x0, lam0, steps=50):
    # Newton's method on the critical-point equations of L(x, lam) = f(x) + lam . g(x)
    n = x0.numel()
    L = lambda z: f(z[:n]) + z[n:] @ g(z[:n])
    z = torch.cat([x0, lam0])
    for _ in range(steps):
        z = z - torch.linalg.solve(hessian(L)(z), jacrev(L)(z))
    return z[:n], z[n:]

2.15.0+cu132


## 1. Lagrange multipliers

Figure 11.1 and Exercise 11.1 (a linear function on a circle), Exercise 11.2
(a linear form on the unit sphere of another inner product), Example 11.3 and
Exercise 11.3 (eigenvalues as multipliers), and the pitfall on the cusp.
At each solution the row `[df]` is a combination of the rows of the Jacobian
matrix of `g`, and it vanishes on the kernel of that matrix.

In [2]:
f = lambda p: p[0] + 2 * p[1]
g = lambda p: (p[0] ** 2 + p[1] ** 2 - 5).reshape(1)
for start, lam0, point, lam, value in ((torch.tensor([1.0, 1.5]), torch.tensor([-1.0]), [1.0, 2.0], -0.5, 5.0),
                                       (torch.tensor([-1.0, -1.5]), torch.tensor([1.0]), [-1.0, -2.0], 0.5, -5.0)):
    x, l = newton_lagrange(f, g, start, lam0)
    print("point", x.tolist(), " lambda", l.item(), " f =", f(x).item())
    assert torch.allclose(x, torch.tensor(point)) and abs(l.item() - lam) < 1e-12 and abs(f(x).item() - value) < 1e-12
    Jg = jacrev(g)(x)
    assert torch.allclose(jacrev(f)(x) + l @ Jg, torch.zeros(2), atol=1e-12)          # stationarity
    tangent = torch.tensor([-x[1], x[0]])                                             # kernel of Jg
    assert abs((Jg @ tangent).item()) < 1e-12 and abs((jacrev(f)(x) @ tangent).item()) < 1e-12
# brute force on the circle
th = torch.linspace(0, 2 * math.pi, 200001)
vals = math.sqrt(5) * (torch.cos(th) + 2 * torch.sin(th))
print("max and min of f on the circle by scanning:", round(vals.max().item(), 6), round(vals.min().item(), 6))
assert abs(vals.max().item() - 5) < 1e-8 and abs(vals.min().item() + 5) < 1e-8

# Exercise 11.2: phi = [1 2] on x^T M x = 1, M = diag(1, 4)
M = torch.diag(torch.tensor([1.0, 4.0])); phi = torch.tensor([1.0, 2.0])
x, l = newton_lagrange(lambda p: phi @ p, lambda p: (p @ M @ p - 1).reshape(1),
                       torch.tensor([0.8, 0.3]), torch.tensor([-0.5]))
print("Exercise 11.2: x =", x.tolist(), " value", (phi @ x).item(), " lambda", l.item())
assert torch.allclose(x, torch.tensor([1.0, 0.5]) / math.sqrt(2)) and abs((phi @ x).item() - math.sqrt(2)) < 1e-12
assert abs(l.item() + 1 / math.sqrt(2)) < 1e-12

# eigenvalues as multipliers
A = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
quad = lambda p: p @ A @ p; sphere = lambda p: (p @ p - 1).reshape(1)
for start in (torch.tensor([0.9, 0.5]), torch.tensor([0.9, -0.5])):
    x, l = newton_lagrange(quad, sphere, start, torch.tensor([-quad(start).item()]))
    print("critical point", [round(v, 4) for v in x.tolist()], " multiplier", round(l.item(), 6),
          " A x + lambda x =", (A @ x + l * x).abs().max().item())
    assert torch.allclose(A @ x, -l * x, atol=1e-12)

# the cusp y^2 = x^3: minimum of x at the origin, where dg = 0
gc = lambda p: p[1] ** 2 - p[0] ** 3
print("cusp: dg(0) =", jacrev(gc)(torch.zeros(2)).tolist(), "  df(0) = [1, 0]: no multiplier exists")
assert jacrev(gc)(torch.zeros(2)).norm() == 0

point [1.0, 2.0]  lambda -0.5  f = 5.0
point [-1.0, -2.0]  lambda 0.5  f = -5.0
max and min of f on the circle by scanning: 5.0 -5.0


Exercise 11.2: x = [0.7071067811865475, 0.35355339059327373]  value 1.414213562373095  lambda -0.7071067811865476
critical point [0.7071, 0.7071]  multiplier -3.0  A x + lambda x = 0.0


critical point [0.7071, -0.7071]  multiplier -1.0  A x + lambda x = 0.0
cusp: dg(0) = [0.0, 0.0]   df(0) = [1, 0]: no multiplier exists


## 2. Multipliers are sensitivities

Proposition 11.4 and Exercise 11.5: the derivative of the optimal value with
respect to the constraint level is `-lambda`. Checked on the exercise and on
a problem without a closed form.

In [3]:
# Exercise 11.5
for c in (0.5, 2.0, -3.0):
    x, l = newton_lagrange(lambda p: p @ p, lambda p: (p[0] + p[1] - c).reshape(1),
                           torch.tensor([0.1, 0.2]), torch.tensor([0.0]))
    assert torch.allclose(x, torch.tensor([c / 2, c / 2])) and abs(l.item() + c) < 1e-12
    assert abs((x @ x).item() - c * c / 2) < 1e-12
print("x + y = c: a(c) = (c/2, c/2), v(c) = c^2/2, lambda = -c, v'(c) = c = -lambda")

f = lambda p: torch.exp(p[0]) + p[1] ** 2 + p[0] * p[2] + p[2] ** 4
gbase = lambda p: torch.stack([p[0] + p[1] ** 2 + p[2] - 1.0, p[0] * p[1] - 0.2])
def value(c):
    x, l = newton_lagrange(f, lambda p: gbase(p) - c, torch.tensor([0.3, 0.6, 0.2]), torch.zeros(2))
    return f(x), l
v0, lam = value(torch.zeros(2))
e = 1e-6
dv = torch.stack([(value(e * torch.eye(2)[i])[0] - value(-e * torch.eye(2)[i])[0]) / (2 * e) for i in range(2)])
print("dv/dc by finite differences:", dv.tolist(), "   -lambda:", (-lam).tolist())
assert torch.allclose(dv, -lam, atol=1e-6)

x + y = c: a(c) = (c/2, c/2), v(c) = c^2/2, lambda = -c, v'(c) = c = -lambda


dv/dc by finite differences: [0.43979302111196716, 2.063748361647022]    -lambda: [0.4397930211854719, 2.063748361747854]


## 3. Second-order conditions

Theorem 11.5: the second differential of the Lagrangian restricted to the
tangent space. Example 11.6 (`y` on the parabola `y = x^2`), Exercise 11.3
(eigenvectors) and Exercise 11.4 (the hyperbola).

In [4]:
def reduced_hessian(f, g, x, lam):
    ell = lambda p: f(p) + lam @ g(p)
    H = hessian(ell)(x)
    Jg = jacrev(g)(x).reshape(lam.numel(), -1)
    _, _, Vh = torch.linalg.svd(Jg)
    Z = Vh[lam.numel():].T                       # basis of the tangent space ker Dg(a)
    return Z.T @ H @ Z, H

f = lambda p: p[1]; g = lambda p: (p[1] - p[0] ** 2).reshape(1)
Q, H = reduced_hessian(f, g, torch.zeros(2), torch.tensor([-1.0]))
print("parabola: D^2 f = 0, D^2 ell =", H.tolist(), " Q on the tangent space =", Q.tolist())
assert torch.allclose(H, torch.diag(torch.tensor([2.0, 0.0]))) and abs(Q.item() - 2) < 1e-12

for vec, lam, q in ((torch.tensor([1.0, -1.0]) / math.sqrt(2), -1.0, 4.0),
                    (torch.tensor([1.0, 1.0]) / math.sqrt(2), -3.0, -4.0)):
    Q, _ = reduced_hessian(quad, sphere, vec, torch.tensor([lam]))
    print("eigenvector", [round(v, 3) for v in vec.tolist()], " Q =", round(Q.item(), 6))
    assert abs(Q.item() - q) < 1e-12

x, l = newton_lagrange(lambda p: p @ p, lambda p: (p[0] * p[1] - 1).reshape(1),
                       torch.tensor([1.2, 0.9]), torch.tensor([-1.0]))
Q, _ = reduced_hessian(lambda p: p @ p, lambda p: (p[0] * p[1] - 1).reshape(1), x, l)
v = torch.tensor([1.0, -1.0])                                   # tangent vector used in the solution
Hl = hessian(lambda p: p @ p + l[0] * (p[0] * p[1] - 1))(x)
print("hyperbola: closest point", x.tolist(), " lambda", l.item(), " Q(v, v) for v = (1, -1):", (v @ Hl @ v).item())
assert torch.allclose(x, torch.ones(2)) and abs(l.item() + 2) < 1e-12
assert abs((v @ Hl @ v).item() - 8) < 1e-12 and abs(Q.item() - 4) < 1e-12   # 8 = 2 * (value on the unit vector)

parabola: D^2 f = 0, D^2 ell = [[2.0, 0.0], [0.0, 0.0]]  Q on the tangent space = [[2.0]]
eigenvector [0.707, -0.707]  Q = 4.0
eigenvector [0.707, 0.707]  Q = -4.0


hyperbola: closest point [1.0, 1.0]  lambda -2.0  Q(v, v) for v = (1, -1): 8.0


## 4. Inequality constraints

Theorem 11.7 on Exercise 11.6 (closest point of the unit disk), with the
sign of the multiplier, complementary slackness, and Theorem 11.8 (in the
convex case the conditions give the global minimum). Then weak and strong
duality on the same problem.

In [5]:
def disk_problem(c):
    f = lambda p: ((p - c) ** 2).sum(); h = lambda p: p @ p - 1
    if c.norm() <= 1:
        return c.clone(), 0.0, f, h
    return c / c.norm(), c.norm().item() - 1, f, h

for c in (torch.tensor([2.0, 1.0]), torch.tensor([0.3, 0.2])):
    a, mu, f, h = disk_problem(c)
    stat = jacrev(f)(a) + mu * jacrev(h)(a)
    print("centre", c.tolist(), " a =", [round(v, 4) for v in a.tolist()], " mu =", round(mu, 4),
          " h(a) =", round(h(a).item() + 0.0, 6), " stationarity residual", stat.abs().max().item())
    assert stat.abs().max() < 1e-12 and mu >= 0 and abs(mu * h(a).item()) < 1e-12
    pts = torch.randn(20000, 2); pts = pts[(pts ** 2).sum(1) <= 1]            # feasible points
    assert (((pts - c) ** 2).sum(1) >= f(a) - 1e-12).all()                     # global minimum
a, mu, f, h = disk_problem(torch.tensor([2.0, 1.0]))
assert torch.allclose(a, torch.tensor([0.894, 0.447]), atol=5e-4) and abs(mu - (math.sqrt(5) - 1)) < 1e-12

# dual function q(mu) = min_x f(x) + mu h(x) = mu |c|^2/(1 + mu) - mu, for mu >= 0
c = torch.tensor([2.0, 1.0]); c2 = (c @ c).item()
q = lambda m: m * c2 / (1 + m) - m
mus = torch.linspace(0, 6, 60001)
qs = mus * c2 / (1 + mus) - mus
print("primal minimum:", f(a).item(), "   largest dual value:", qs.max().item(), " at mu =", mus[qs.argmax()].item())
assert (qs <= f(a).item() + 1e-12).all()                                       # weak duality
assert abs(qs.max().item() - f(a).item()) < 1e-6 and abs(mus[qs.argmax()].item() - mu) < 1e-3
# check the closed form of q against a direct minimization at one value
m = 0.7; xm = c / (1 + m)
assert abs((f(xm) + m * h(xm)).item() - q(m)) < 1e-12

centre [2.0, 1.0]  a = [0.8944, 0.4472]  mu = 1.2361  h(a) = -0.0  stationarity residual 0.0
centre [0.3, 0.2]  a = [0.3, 0.2]  mu = 0.0  h(a) = -0.87  stationarity residual 0.0
primal minimum: 1.527864045000421    largest dual value: 1.5278640445418366  at mu = 1.2361


## 5. Projection and projected gradient descent

Proposition 11.9 (the projection is characterized by an inequality and is
1-Lipschitz), Exercise 11.7 and Theorem 11.11 (rate of projected gradient
descent).

In [6]:
clip = lambda p: p.clamp(0.0, 1.0)                            # projection on the square [0, 1]^2
ball = lambda p: p / max(1.0, p.norm().item())                # projection on the unit ball
torch.manual_seed(1)
for P in (clip, ball):
    for _ in range(200):
        a, b = 2 * torch.randn(2), 2 * torch.randn(2)
        assert (P(a) - P(b)).norm() <= (a - b).norm() + 1e-12
        c = P(2 * torch.randn(2))                             # a point of C
        assert (a - P(a)) @ (c - P(a)) <= 1e-12
print("projections on the square and on the ball: <a - p, c - p> <= 0 and 1-Lipschitz, on 200 samples each")

f7 = lambda p: (p[0] - 2) ** 2 + 4 * (p[1] - 1) ** 2
x = torch.zeros(2); path = [x.clone()]
for _ in range(6):
    x = clip(x - jacrev(f7)(x) / 8); path.append(x.clone())
print("Exercise 11.7:", [[round(v, 4) for v in p.tolist()] for p in path[:5]])
assert [p.tolist() for p in path[:5]] == [[0.0, 0.0], [0.5, 1.0], [0.875, 1.0], [1.0, 1.0], [1.0, 1.0]]
astar = path[-1]; gam = jacrev(f7)(astar)
assert all((gam @ (clip(torch.rand(2)) - astar)).item() >= -1e-12 for _ in range(500))
print("d f(a*) =", gam.tolist(), " multiplier of x1 <= 1:", -gam[0].item())
assert gam.tolist() == [-2.0, 0.0]

# rate: f with mu = 1, L = 4 (Exercise 9.6), projected on a ball that excludes the free minimum
f6 = lambda p: 0.5 * ((p - torch.tensor([3.0, 2.0])) ** 2).sum() + 3 * torch.log(torch.cosh(p[1] - 2.0))
kap = lambda eta: max(abs(1 - eta), abs(1 - 4 * eta))
for eta in (0.25, 0.4):
    x = torch.tensor([-0.5, 0.3]); pts = [x.clone()]
    for _ in range(200):
        x = ball(x - eta * jacrev(f6)(x)); pts.append(x.clone())
    star = pts[-1]
    assert torch.allclose(star, ball(star - eta * jacrev(f6)(star)), atol=1e-13)      # fixed point
    errs = [(p - star).norm().item() for p in pts[:40]]
    assert all(errs[k] <= kap(eta) ** k * errs[0] + 1e-13 for k in range(40))
    print(f"eta = {eta}: limit {[round(v, 5) for v in star.tolist()]} on the unit circle,"
          f" bound kappa = {kap(eta):.2f}, observed first ratios {[round(errs[k + 1] / errs[k], 3) for k in range(3)]}")

projections on the square and on the ball: <a - p, c - p> <= 0 and 1-Lipschitz, on 200 samples each
Exercise 11.7: [[0.0, 0.0], [0.5, 1.0], [0.875, 1.0], [1.0, 1.0], [1.0, 1.0]]
d f(a*) = [-2.0, 0.0]  multiplier of x1 <= 1: 2.0
eta = 0.25: limit [0.55707, 0.83047] on the unit circle, bound kappa = 0.75, observed first ratios [0.281, 0.333, 0.325]


eta = 0.4: limit [0.55707, 0.83047] on the unit circle, bound kappa = 0.60, observed first ratios [0.157, 0.18, 0.175]


## 6. Maximum entropy and the softmax

The machine-learning box of Section 11.5 and Exercise 11.8: with energies
`(0, 1, 2)` and mean `1/2`, the maximum-entropy distribution is a softmax
`p ~ exp(beta E)`.

In [7]:
E = torch.tensor([0.0, 1.0, 2.0])
u = (math.sqrt(13) - 1) / 6; beta = math.log(u)
p = torch.softmax(beta * E, 0)
print("u =", round(u, 4), " beta =", round(beta, 4), " p =", [round(v, 4) for v in p.tolist()], " mean =", (p @ E).item())
assert abs((p @ E).item() - 0.5) < 1e-12 and torch.allclose(p, torch.tensor([0.616, 0.268, 0.116]), atol=5e-4)

# the Lagrange conditions: dH + lam0 d(sum p) + beta d(sum p E) = 0
H = lambda q: -(q * torch.log(q)).sum()
lam0 = 1 + math.log(p[0].item())                                   # from the component with E = 0
resid = jacrev(H)(p) + lam0 * torch.ones(3) + beta * E
print("stationarity residual:", resid.abs().max().item())
assert resid.abs().max() < 1e-12

# no feasible distribution has a larger entropy: feasible set is a segment p + t d
d = torch.tensor([1.0, -2.0, 1.0])                                 # sum d = 0 and d . E = 0
ts = torch.linspace(-0.11, 0.13, 2401)
ent = torch.stack([H(p + t * d) for t in ts])
print("entropy at the solution:", H(p).item(), "  largest on the feasible segment:", ent.max().item())
assert ent.max() <= H(p) + 1e-12

u = 0.4343  beta = -0.8341  p = [0.6162, 0.2676, 0.1162]  mean = 0.5
stationarity residual: 1.1102230246251565e-16
entropy at the solution: 0.9012347006341614   largest on the feasible segment: 0.9012347006341614


## 7. The support vector machine

Exercise 11.9, then a two-dimensional example without bias term: the dual
problem `max sum(alpha) - (1/2) alpha^T K alpha`, `alpha >= 0`, is solved by
projected gradient ascent, and the conditions of Theorem 11.7 are checked:
the weight is a combination of the data, the multipliers are non-negative
and vanish off the margin, and the dual value equals the primal value.

In [8]:
# Exercise 11.9: x = (1, -1), y = (+1, -1): w = 1, b = 0, alpha = (1/2, 1/2)
w, b, alpha = 1.0, 0.0, torch.tensor([0.5, 0.5])
xs, ys = torch.tensor([1.0, -1.0]), torch.tensor([1.0, -1.0])
assert abs((alpha * ys * xs).sum().item() - w) < 1e-12 and abs((alpha * ys).sum().item()) < 1e-12
assert torch.allclose(ys * (w * xs + b), torch.ones(2))
assert abs(alpha.sum().item() - 0.5 * w * w - 0.5 * w * w) < 1e-12        # dual value = primal value = 1/2
print("third point x = 3, label +1: y * l(x) =", 3 * w + b, "> 1, inactive")

torch.manual_seed(2)
n = 40
ysep = torch.cat([torch.ones(n // 2), -torch.ones(n // 2)])
X = torch.randn(n, 2) + 2.2 * ysep[:, None] * torch.tensor([1.0, 0.6])
K = (X @ X.T) * torch.outer(ysep, ysep)                                 # y_i y_j <x_i, x_j>
dual = lambda al: al.sum() - 0.5 * al @ K @ al
step = 1.0 / torch.linalg.eigvalsh(K)[-1].item()
al = torch.zeros(n)
for _ in range(20000):
    al = (al + step * (1 - K @ al)).clamp(min=0.0)                      # ascent step, then projection
w = (al * ysep) @ X
margins = ysep * (X @ w)
support = al > 1e-8
print("support vectors:", int(support.sum()), "of", n, "  smallest margin:", round(margins.min().item(), 6))
print("primal value |w|^2/2 =", round(0.5 * (w @ w).item(), 6), "   dual value =", round(dual(al).item(), 6))
assert (al >= 0).all() and margins.min() >= 1 - 1e-6                    # dual and primal feasibility
assert torch.allclose(margins[support], torch.ones(int(support.sum())), atol=1e-6)   # complementary slackness
assert abs(0.5 * (w @ w).item() - dual(al).item()) < 1e-6               # strong duality
for _ in range(2000):                                                    # weak duality at random alpha >= 0
    assert dual(torch.rand(n) * 0.2).item() <= 0.5 * (w @ w).item() + 1e-9

third point x = 3, label +1: y * l(x) = 3.0 > 1, inactive


support vectors: 2 of 40   smallest margin: 1.0
primal value |w|^2/2 = 1.057569    dual value = 1.057569


## 8. Constraints and penalties

Minimizing `f + (rho/2) |g|^2` approaches the constrained solution as `rho`
grows; `rho * g(a_rho)` approaches the multiplier, the violation decreases
like `1/rho`, and the condition number of the penalized problem grows like
`rho`.

In [9]:
f = lambda p: (p[0] - 2) ** 2 + 2 * (p[1] - 1) ** 2 + p[0] * p[1]
g = lambda p: (p[0] + 2 * p[1] - 1).reshape(1)
xc, lam = newton_lagrange(f, g, torch.zeros(2), torch.zeros(1))
print("constrained solution", [round(v, 6) for v in xc.tolist()], " lambda =", round(lam.item(), 6))
prev = None
for rho in (1.0, 10.0, 100.0, 1000.0):
    pen = lambda p: f(p) + 0.5 * rho * (g(p) ** 2).sum()
    x = torch.zeros(2)
    for _ in range(30):
        x = x - torch.linalg.solve(hessian(pen)(x), jacrev(pen)(x))
    ev = torch.linalg.eigvalsh(hessian(pen)(x))
    viol = abs(g(x).item())
    print(f"rho = {rho:7.1f}: violation {viol:.2e}  rho*g = {rho * g(x).item():+.5f}"
          f"  distance to the solution {(x - xc).norm().item():.2e}  condition number {ev[-1].item() / ev[0].item():.1f}")
    if prev is not None:
        assert 5 < prev / viol < 12                     # violation ~ 1/rho
    prev = viol
assert abs(rho * g(x).item() - lam.item()) < 1e-2 * abs(lam.item())

constrained solution [1.25, -0.125]  lambda = 1.625
rho =     1.0: violation 8.67e-01  rho*g = +0.86667  distance to the solution 3.91e-01  condition number 5.9


rho =    10.0: violation 1.49e-01  rho*g = +1.49425  distance to the solution 6.73e-02  condition number 34.0


rho =   100.0: violation 1.61e-02  rho*g = +1.61090  distance to the solution 7.26e-03  condition number 315.3


rho =  1000.0: violation 1.62e-03  rho*g = +1.62358  distance to the solution 7.32e-04  condition number 3127.8


## Summary

* At a constrained extremum the row `[df]` is a combination of the rows of
  the constraint Jacobian; the coefficients are the multipliers, and minus
  the multipliers are the derivatives of the optimal value with respect to
  the constraint levels.
* The sign of the reduced Hessian of the Lagrangian classifies constrained
  critical points.
* For inequalities the multipliers are non-negative and vanish on inactive
  constraints; in the convex case the dual value equals the primal value.
* Projected gradient descent converges at the rate of gradient descent.
* The softmax is a maximum-entropy distribution; the support vectors are
  the points with a positive multiplier.